# Build a small RNN to generate smiles codes

Try on two different datasets: a small toy dataset and the smiles column from the ESOL dataset (random sample of 300 points - training time estimated to be ~2-5 min - decrease the sample number if you had issues with runtime before).

In [6]:
import pandas as pd

smiles_list = [
    "CCO",
    "CCN",
    "C=O",
    "C1CCCCC1",
    "CC(=O)O",
    "CCC",
    "COC",
    "CCCl",
    "O"
]

df = pd.read_csv("esol_modified.csv").dropna(subset=["SMILES"]).sample(300)
smiles_list2 = df["SMILES"].to_list()
smiles_list

['CCO', 'CCN', 'C=O', 'C1CCCCC1', 'CC(=O)O', 'CCC', 'COC', 'CCCl', 'O']

Build vocabulary and tokeniser

In [7]:
chars = sorted(list(set("".join(smiles_list))))
stoi = {ch: i for i, ch in enumerate(chars)} #string to index
itos = {i: ch for ch, i in stoi.items()} # index to string

vocab_size = len(chars)

In [8]:
print("stoi: ", stoi)
print("itos:", itos)

stoi:  {'(': 0, ')': 1, '1': 2, '=': 3, 'C': 4, 'N': 5, 'O': 6, 'l': 7}
itos: {0: '(', 1: ')', 2: '1', 3: '=', 4: 'C', 5: 'N', 6: 'O', 7: 'l'}


In [9]:
def encode(smile):
    return [stoi[c] for c in smile]

data = [encode(s) for s in smiles_list]

In [10]:
data

[[4, 4, 6],
 [4, 4, 5],
 [4, 3, 6],
 [4, 2, 4, 4, 4, 4, 4, 2],
 [4, 4, 0, 3, 6, 1, 6],
 [4, 4, 4],
 [4, 6, 4],
 [4, 4, 4, 7],
 [6]]

In [11]:
data = [seq for seq in data if len(seq) > 1]

Build Model

In [12]:
import torch
import torch.nn as nn

class SmilesRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x) # two values are returned by RNN: output and hidden state (not used)
        out = self.fc(out)
        return out

Train the RNN

In [13]:
model = SmilesRNN(vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(200):
    total_loss = 0

    for seq in data:
        x = torch.tensor(seq[:-1], dtype=torch.long).unsqueeze(0)
        y = torch.tensor(seq[1:]).unsqueeze(0)

        output = model(x)
        loss = criterion(output.view(-1, vocab_size), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.3f}")

Epoch 0, Loss: 15.530
Epoch 20, Loss: 6.638
Epoch 40, Loss: 6.722
Epoch 60, Loss: 6.462
Epoch 80, Loss: 6.296
Epoch 100, Loss: 6.213
Epoch 120, Loss: 6.154
Epoch 140, Loss: 6.122
Epoch 160, Loss: 6.097
Epoch 180, Loss: 6.086


In [14]:
import random

def generate(model, start_char='C', max_len=8):
    model.eval()
    
    idx = torch.tensor([[stoi[start_char]]])
    result = start_char

    for i in range(max_len):
        output = model(idx)
        probs = torch.softmax(output[0, -1], dim=0)
        
        idx_next = torch.multinomial(probs, 1).item()
        char = itos[idx_next]

        result += char
        idx = torch.tensor([[idx_next]])

    return result

for i in range(10):
    print(generate(model))

CCOCC=OCC
CCO)OC=O)
CCCC=OCCC
COC=OCCCC
CCCCOCOC=
CCOCC=OCC
COCC=O)OC
CC=O)OC=O
CCCCCO)OC
CC=OCCC=O
